In [1]:
import pandas as pd
import numpy as np

TARGETS = ["X"]

SETS = ["Train_1", "Train_2", "Val_1", "Val_2", "Test_1", "Test_2", "LSG_1", "LSG_2"]

results = pd.read_excel("resultados.xlsx")


In [2]:

results = pd.read_excel("resultados.xlsx")

best_models_tables = {}

N = 10  # top modelos

for target in TARGETS:

    # 🔹 nomes das colunas
    r2_cols = {s: f"R2_{s}_{target}" for s in SETS}

    # 🔹 garantir que só usamos colunas existentes
    valid_r2 = {k: v for k, v in r2_cols.items() if v in results.columns}


    df = results.copy()# 🔹 pegar todas as colunas de R2 válidas
    r2_all_cols = list(valid_r2.values())

    # 🔹 remover linhas onde QUALQUER R2 < 0
    df = df[(df[r2_all_cols] >= 0.0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    
    train_cols = [v for k, v in valid_r2.items() if "Train" in k]
    test_cols  = [v for k, v in valid_r2.items() if "Test" in k]
    val_cols   = [v for k, v in valid_r2.items() if "Val" in k]
    test2_cols  = [v for k, v in valid_r2.items() if "LSG" in k]


    df["R2_train_mean"] = df[train_cols].mean(axis=1)
    df["R2_test_mean"]  = df[test_cols].mean(axis=1)
    df["R2_val"]        = df[val_cols].mean(axis=1)
    df["R2_test2_mean"]        = df[test2_cols].mean(axis=1)

    # =========================
    # 🔹 SCORE
    # =========================
    
    w_val = 0.3
    w_test = 0.2
    w_test2 = 0.3
    w_train = 0.2

    metric_cols = list(r2_cols.values())
    df["R2_std"] = df[metric_cols].std(axis=1)

    df["Score"] = (
        w_val * df["R2_val"] +
        w_test * df["R2_test_mean"] +
        w_test2 * df["R2_test2_mean"] +
        w_train * df["R2_train_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    
    df_sorted = df.sort_values(by="Score", ascending=False)

    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_val", "R2_test_mean", "R2_test2_mean", "R2_train_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    # 🔹 organizar colunas
    metric_cols = []

    for s in SETS:
        if s in valid_r2:
            metric_cols.append(valid_r2[s])

    final_cols = ["model", "Neurons"] + metric_cols + [
        "R2_val", "R2_test_mean", "R2_test2_mean", "R2_train_mean", "Score"
    ]

    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - X


,model,Neurons,R2_val,R2_test_mean,R2_test2_mean,R2_train_mean,Score
178,model_arch64-32_r0.9_Ld0.7_Lp0.3_seed3227,"[64, 32]",0.964804,0.790423,0.533384,0.980800,0.784097
255,model_arch32-16-8_r0.9_Ld0.7_Lp0.3_seed5848,"[32, 16, 8]",0.851539,0.176962,0.264279,0.991141,0.528648
228,model_arch16-8-4_r0.9_Ld0.3_Lp0.7_seed3227,"[16, 8, 4]",0.677637,0.087584,0.350048,0.940504,0.476878



📊 MÉTRICAS COMPLETAS - TOP 10 (X)


,model,Neurons,R2_Train_1_X,R2_Train_2_X,R2_Val_1_X,R2_Val_2_X,R2_Test_1_X,R2_Test_2_X,R2_LSG_1_X,R2_LSG_2_X,R2_val,R2_test_mean,R2_test2_mean,R2_train_mean,Score
178,model_arch64-32_r0.9_Ld0.7_Lp0.3_seed3227,"[64, 32]",0.969237,0.992363,0.967784,0.961824,0.738450,0.842397,0.490379,0.576388,0.964804,0.790423,0.533384,0.980800,0.784097
255,model_arch32-16-8_r0.9_Ld0.7_Lp0.3_seed5848,"[32, 16, 8]",0.988417,0.993865,0.727553,0.975526,0.004883,0.349041,0.315062,0.213495,0.851539,0.176962,0.264279,0.991141,0.528648
228,model_arch16-8-4_r0.9_Ld0.3_Lp0.7_seed3227,"[16, 8, 4]",0.890357,0.990651,0.866083,0.489190,0.144685,0.030483,0.202982,0.497115,0.677637,0.087584,0.350048,0.940504,0.476878


In [3]:
final_table.to_excel("BestModels.xlsx")